# 00_feature_audit: which columns are safe to use

**Owner:** Liza

**Covers:** Step 3 in the README: decision moment, column sorting, the four leakage tests (arithmetic, single-column power, defaulters vs repayers, proxy), and the feature-set comparison (F0–F3). Final table goes in `learnings/03_feature_audit.md`.

**Team rules for this notebook**

1. **Explain everything as plainly as possible.** The rest of the group will not redo this part; they will rely on what you write here. Every step gets a markdown cell before it: what we are doing, why, and what the result means in plain English. No unexplained jargon.
2. **Test every reasonable alternative.** Do not run one option and move on. For each step, try the alternatives, put the results side by side in a table, and state which one wins and why.
3. **It must run top to bottom** (Kernel > Restart & Run All) from a fresh clone with `data/` in place.

In [1]:
import pandas as pd
import numpy as np

DATA = '../data/zindi/'

## Step 1: Get to know the 16 columns

**What we are doing:** loading the past-loans spreadsheet (`Train.csv`) and making a summary of every column.

**Why:** before deciding whether the model may use a column, we need to know what it is and what its values look like.

**The table below shows, for each column:**
- **meaning**: the official description from Zindi (`VariableDefinitions.txt`);
- **type**: number, text, or date;
- **unique values**: how many different values it has (e.g. `country_id` has 1, meaning every row is the same);
- **missing**: how many empty cells;
- **min / max**: the smallest and largest value (for numbers and dates).

We only print summaries, never individual rows, because the Zindi data is private.

In [2]:
train = pd.read_csv(DATA + 'Train.csv', parse_dates=['disbursement_date', 'due_date'])
print('Rows (loans):', len(train))
print('Columns:', train.shape[1])

Rows (loans): 68654
Columns: 16


In [3]:
# The official meaning of each column, copied from VariableDefinitions.txt
meaning = {
    'ID': 'Unique identifier for each row',
    'customer_id': 'Unique identifier for each customer',
    'country_id': 'Country where the loan was issued',
    'tbl_loan_id': 'Unique identifier for each loan',
    'lender_id': 'The lender or institution that issued the loan',
    'loan_type': 'The category or type of loan',
    'Total_Amount': 'Total loan amount initially disbursed',
    'Total_Amount_to_Repay': 'Total the customer must repay (principal + interest + fees)',
    'disbursement_date': 'Date the money was sent to the customer',
    'due_date': 'Date by which the loan must be repaid',
    'duration': 'Length of the loan in days',
    'New_versus_Repeat': "Customer's first loan ('New') or not ('Repeat')",
    'Amount_Funded_By_Lender': 'Portion of the loan funded by the lender',
    'Lender_portion_Funded': 'Percentage of the loan funded by the lender',
    'Lender_portion_to_be_repaid': 'Portion of the outstanding loan to be repaid to the lender',
    'target': '1 = customer defaulted (did not pay), 0 = paid',
}

def kind(col):
    if pd.api.types.is_datetime64_any_dtype(train[col]):
        return 'date'
    if pd.api.types.is_numeric_dtype(train[col]):
        return 'number'
    return 'text'

inventory = pd.DataFrame({
    'meaning': [meaning[c] for c in train.columns],
    'type': [kind(c) for c in train.columns],
    'unique values': train.nunique().values,
    'missing': train.isna().sum().values,
    'min': [train[c].min() if kind(c) != 'text' else '' for c in train.columns],
    'max': [train[c].max() if kind(c) != 'text' else '' for c in train.columns],
}, index=train.columns)
pd.set_option('display.max_colwidth', 70)
inventory

,meaning,type,unique values,missing,min,max
ID,Unique identifier for each row,text,68654,0,,
customer_id,Unique identifier for each customer,number,6540,0,145,312737
country_id,Country where the loan was issued,text,1,0,,
tbl_loan_id,Unique identifier for each loan,number,66520,0,101323,375320
lender_id,The lender or institution that issued the loan,number,4,0,245684,267278
loan_type,The category or type of loan,text,22,0,,
Total_Amount,Total loan amount initially disbursed,number,19076,0,2.0,23000000.0
Total_Amount_to_Repay,Total the customer must repay (principal + interest + fees),number,21920,0,0.0,25415000.0
disbursement_date,Date the money was sent to the customer,date,768,0,2021-10-04 00:00:00,2024-11-14 00:00:00
due_date,Date by which the loan must be repaid,date,893,0,2021-10-21 00:00:00,2025-10-27 00:00:00


### Some "numbers" are really name tags

`customer_id`, `tbl_loan_id` and `lender_id` contain only digits, so the computer calls them numbers. But they are **labels**: customer 300,000 is not "bigger" than customer 145. The model must never treat them as amounts.

`ID` is text because it starts with `ID_`. It looks like customer + loan + lender glued together. The check below tests that on every row.

In [4]:
glued = 'ID_' + train.customer_id.astype(str) + train.tbl_loan_id.astype(str) + train.lender_id.astype(str)
print(f'Rows where ID = customer + loan + lender: {(glued == train.ID).mean():.2%}')

Rows where ID = customer + loan + lender: 100.00%


### How the loans are split into groups

This counts how many loans fall into each group, like counting the students in each class. It matters for three reasons:

1. **Some groups are tiny.** Only a few hundred loans are "New" (first loans). With so few examples, the model will struggle to learn anything reliable about first-time borrowers.
2. **Some groups dominate.** One lender gave out most of the loans, so what the model learns will mostly be about that lender.
3. **These groups are used later for fairness** (step 11). Group sizes tell us which fairness checks we can trust: a check on a tiny group is shaky.

In [5]:
for col in ['country_id', 'New_versus_Repeat', 'lender_id']:
    print(train[col].value_counts().to_string(), '\n')
print('loan_type: ', train['loan_type'].nunique(), 'types; the 5 most common:')
print(train['loan_type'].value_counts().head().to_string())

country_id
Kenya    68654 

New_versus_Repeat
Repeat Loan    68087
New Loan         567 

lender_id
267278    64653
251804     3542
267277      271
245684      188 

loan_type:  22 types; the 5 most common:
loan_type
Type_1     61723
Type_7      2790
Type_5      1521
Type_4      1235
Type_10      466


### How many loans were not paid back?

In [6]:
counts = train['target'].value_counts()
print('Paid back (0):    ', counts[0])
print('Not paid back (1):', counts[1])
print(f'Share not paid back: {train.target.mean():.2%}')

Paid back (0):     67396
Not paid back (1): 1258
Share not paid back: 1.83%


## Step 2: The timeline of one loan

**What we are doing:** drawing the life of a loan, and marking the moment NkwaPay has to decide.

```
 1. Customer       2. NkwaPay      3. Money sent          4. Due date        5. Outcome
    applies  ───►     decides  ───►   (disbursement_date) ───► (due_date) ───►  paid (target=0)
                         ▲                                                    or not (target=1)
                         │
               THE DECISION MOMENT
```

**The decision moment** is just before the money is sent. The column closest to it is `disbursement_date`. (The data has no application or approval date.)

**The rule for every column:** *would NkwaPay know this value at the decision moment?*
- Yes → the model may use it.
- No (it was filled in later) → it "peeks at the answer" and must be dropped.

**Why the dates matter:** if `due_date` is simply `disbursement_date + duration`, it is known on day 1 and is safe. The check below tests that on every row.

In [7]:
gap_days = (train.due_date - train.disbursement_date).dt.days
difference = gap_days - train.duration

print(f'Rows where due_date = disbursement_date + duration exactly: {(difference == 0).mean():.2%}')
if (difference != 0).any():
    print('How far off are the other rows? (days: number of loans)')
    print(difference[difference != 0].value_counts().head(10).to_string())

Rows where due_date = disbursement_date + duration exactly: 100.00%


**What this means:** on every single loan, `due_date` is exactly `disbursement_date + duration`. So the due date is fixed on the day the money is sent. It is **known at the decision moment** and is not a leak. It is also not new information: it repeats the other two columns.

### How long are the loans?

The brief says loans are usually 7 to 90 days. This checks that.

In [8]:
bands = pd.cut(train.duration, [0, 7, 14, 30, 90, 2000],
               labels=['1–7 days', '8–14 days', '15–30 days', '31–90 days', 'over 90 days'])
summary = pd.DataFrame({'loans': bands.value_counts(sort=False)})
summary['share'] = (summary.loans / len(train)).map('{:.1%}'.format)
summary

,loans,share
duration,,
1–7 days,65041,94.7%
8–14 days,1572,2.3%
15–30 days,1147,1.7%
31–90 days,742,1.1%
over 90 days,152,0.2%


**What this means:** about 95% of loans last a week or less. Only 0.2% last more than 90 days, so the brief's "7 to 90 days" holds for almost every loan. The few very long loans are rare, unusual cases; we keep them for now but note them.

## Step 3: What exactly does "did not pay back" mean?

**What we are doing:** defining the label, the `target` column the model learns to predict.

**What Zindi tells us:** `target` = 1 means "the customer defaulted on the loan", 0 means "the customer paid the loan". It does **not** say *by when*: on the due date? a week later? a month later?

**Why it matters:** if we don't know when the answer was recorded, we can't say what the model is actually predicting.

**The puzzle:** some loans were sent in the last weeks of the data (Nov 2024) and are due in 2025. If their `target` is already filled in, the answers must have been recorded long after the last loan. The checks below look at this.

In [9]:
last_send = train.disbursement_date.max()
not_yet_due = train.due_date > last_send
print('Last day any loan was sent:', last_send.date())
print('Latest due date in the data:', train.due_date.max().date())
print()
print(f'Loans still not due on that last day: {not_yet_due.sum()} ({not_yet_due.mean():.1%})')
print(f'  share not paid back: {train.target[not_yet_due].mean():.2%}')
print(f'Loans already due:                    {(~not_yet_due).sum()}')
print(f'  share not paid back: {train.target[~not_yet_due].mean():.2%}')

Last day any loan was sent: 2024-11-14
Latest due date in the data: 2025-10-27

Loans still not due on that last day: 190 (0.3%)
  share not paid back: 0.00%
Loans already due:                    68464
  share not paid back: 1.84%


**Zero unpaid loans among the 190 not-yet-due loans. Is that just luck?**

We check how many of them we would *expect* to be unpaid, using the unpaid share of loans of the same length that *were* already due.

In [10]:
already_due = train[~not_yet_due]
rate_by_length = already_due.groupby('duration').target.mean()
expected = train.duration[not_yet_due].map(rate_by_length).fillna(already_due.target.mean()).sum()
print(f'Expected unpaid among the 190: about {expected:.0f}.  Actually unpaid: {train.target[not_yet_due].sum()}')

Expected unpaid among the 190: about 5.  Actually unpaid: 0


**What this means:** we would expect about 5 of these 190 loans to be unpaid, but there are 0. The likely reason: the answers were collected around the last loan date, so these loans **hadn't had the chance to go unpaid yet**. Their `0` means "not unpaid *yet*", not "paid back".

**Decision:** these 190 loans (0.3% of the data) have untrustworthy answers, so we **leave them out when training**. Because there are so few of them, this barely changes anything else.

### Does the share of unpaid loans change over time?

If answers were recorded on a fixed day, the newest loans would have had less time to go wrong, and their default share would look lower (or be missing).

In [11]:
by_quarter = train.groupby(train.disbursement_date.dt.to_period('Q')).target.agg(['count', 'mean'])
by_quarter.columns = ['loans', 'share not paid back']
by_quarter['share not paid back'] = by_quarter['share not paid back'].map('{:.2%}'.format)
by_quarter

,loans,share not paid back
disbursement_date,,
2021Q4,39,53.85%
2022Q1,585,18.29%
2022Q2,479,13.99%
2022Q3,40554,1.00%
2022Q4,22787,1.12%
2023Q1,347,11.24%
2023Q2,182,14.84%
2023Q3,372,13.44%
2023Q4,339,7.96%


**What this means:** the unpaid share jumps around a lot over time, from 1% to over 50%. Almost all loans (63,000 of 68,654) were sent in two quarters of 2022, and those loans were rarely unpaid (about 1%). The other quarters have only a few hundred loans each, with much higher unpaid shares. So the *mix* of loans changes over time. This will matter later when we test the model on newer loans (step 5).

### Do longer loans go unpaid more often?

A longer loan has more time for things to go wrong.

In [12]:
by_length = train.groupby(bands, observed=True).target.agg(['count', 'mean'])
by_length.columns = ['loans', 'share not paid back']
by_length['share not paid back'] = by_length['share not paid back'].map('{:.2%}'.format)
by_length

,loans,share not paid back
duration,,
1–7 days,65041,1.22%
8–14 days,1572,8.21%
15–30 days,1147,13.25%
31–90 days,742,15.90%
over 90 days,152,44.74%


**What this means:** the longer the loan, the more often it goes unpaid: about 1 in 80 for one-week loans, and nearly 1 in 2 for loans longer than 90 days. So `duration` is likely to be a useful clue.

### The label, defined

> **`target` = 1 (default):** the customer did not repay the loan as agreed, as recorded when the data was collected (around mid-November 2024).
> **`target` = 0:** the loan was repaid, or had not gone unpaid by that time.
>
> **Period:** from the day the money is sent until data collection. Zindi does not say whether there is a grace period after the due date. Loans not yet due at collection (190) are left out of training because their answer isn't known yet.

## Step 4: Testing the money columns

There are five money columns. We check them with two simple tests:

- **Test A, the maths check:** is a column just a calculation from other columns? If so, it adds nothing new, and if it is calculated from the loan amount it would give away the Task 1 answer.
- **Test B, defaulters vs repayers:** do people who did not pay back have values that could only be known *after* the loan went wrong? (For example, a late-payment fine.)

Short names used below:

| Short name | Column | Meaning |
|---|---|---|
| amount | `Total_Amount` | money lent |
| repay | `Total_Amount_to_Repay` | total the customer must pay back |
| funded | `Amount_Funded_By_Lender` | money put in by this lender |
| share | `Lender_portion_Funded` | this lender's share of the loan |
| to_be_repaid | `Lender_portion_to_be_repaid` | what must be paid back to this lender |

### Test A: the maths check

We count the share of loans where each formula holds (allowing a 1% rounding difference).

In [13]:
amount = train.Total_Amount
repay = train.Total_Amount_to_Repay
funded = train.Amount_Funded_By_Lender
share = train.Lender_portion_Funded
to_be_repaid = train.Lender_portion_to_be_repaid

def holds(a, b):
    # True where a and b are within 1% of each other
    return np.abs(a - b) <= 0.01 * np.maximum(np.abs(b), 1)

maths = pd.Series({
    'share = funded ÷ amount': holds(share, funded / amount).mean(),
    'to_be_repaid = funded × (repay ÷ amount)': holds(to_be_repaid, funded * repay / amount).mean(),
    'repay is at least amount (customer pays back no less than they borrowed)': (repay >= amount).mean(),
}).map('{:.1%}'.format).to_frame('loans where it holds')
maths

,loans where it holds
share = funded ÷ amount,100.0%
to_be_repaid = funded × (repay ÷ amount),96.2%
repay is at least amount (customer pays back no less than they borrowed),99.9%


**What this means:**
- **share** is *exactly* funded ÷ amount on every loan. It is a calculation, not new information.
- **to_be_repaid** is funded × (repay ÷ amount) on almost every loan: the lender's slice of what the customer owes. So it is built from **repay**, and whatever is wrong with repay is wrong with it too.
- **funded** and **share** are both built from **amount**, which is the Task 1 answer (the loan amount). Giving them to the Task 1 model would hand it the answer.

### Test B: defaulters vs repayers

The **price** of a loan is repay ÷ amount. A price of 1.10 means the customer pays back 10% more than they borrowed.

If the price were set on the day the loan is given, defaulters and repayers taking the *same kind of loan* should pay similar prices. We compare them on the most common loan: Type_1, 7 days, same lender (about 90% of all loans).

In [14]:
train['price'] = repay / amount
same_kind = train[(train.loan_type == 'Type_1') & (train.duration == 7)]
print('Loans of this kind:', len(same_kind), '| lenders:', same_kind.lender_id.nunique())

compare = same_kind.groupby('target').price.agg(['count', 'median', 'min', 'max'])
compare['pays exactly what was borrowed (price = 1)'] = same_kind.groupby('target').price.apply(lambda p: (p == 1).mean())
compare.index = ['repaid (0)', 'not repaid (1)']
compare.round(3)

Loans of this kind: 61723 | lenders: 1


,count,median,min,max,pays exactly what was borrowed (price = 1)
repaid (0),61203,1.007,0.0,1.714,0.392
not repaid (1),520,1.412,1.0,2.181,0.002


In [15]:
from sklearn.metrics import roc_auc_score
print(f'How well price alone separates defaulters from repayers (ROC-AUC): {roc_auc_score(same_kind.target, same_kind.price):.3f}')
print('(0.5 = no better than a coin flip, 1.0 = perfect)')

How well price alone separates defaulters from repayers (ROC-AUC): 0.991
(0.5 = no better than a coin flip, 1.0 = perfect)


**What this means: this is a leak.**

For the *same* kind of loan from the *same* lender:
- people who repaid owe about **1%** extra;
- people who did not repay owe about **41%** extra.

Price alone picks out defaulters almost perfectly (ROC-AUC 0.99). No real price, set on day 1, could predict the future that well. The likely explanation: **late fees and penalties were added to `Total_Amount_to_Repay` after the customer failed to pay.** So the column was updated after the loan was issued, and it "peeks at the answer".

**Decision:** `Total_Amount_to_Repay` is **not available at decision time** → dropped. `Lender_portion_to_be_repaid` is built from it → dropped too.

In [16]:
print('Share of loans where the lender put in nothing (funded = 0):')
print(train.groupby('target').Amount_Funded_By_Lender.apply(lambda f: (f == 0).mean()).map('{:.1%}'.format)
      .rename(index={0: 'repaid (0)', 1: 'not repaid (1)'}).to_string())

Share of loans where the lender put in nothing (funded = 0):
target
repaid (0)        12.6%
not repaid (1)     0.0%


**What this means:** about 1 in 8 repaid loans show the lender putting in nothing, but this **never** happens for a defaulted loan. That pattern is suspicious: a zero could mean the lender's portion was "settled" or reset after repayment. We can't prove it from the data, so we keep `funded` and `share` as **suspects** and test them in the next step (how strongly each column alone predicts default).

## Step 5: How well does each column predict default on its own?

**The idea:** a real clue known on day 1 can help a bit, but it can't be close to perfect, because nobody can see the future. If a single column picks out defaulters almost perfectly, it is probably the answer sheet in disguise.

**How we test each column, fairly:**
1. Split the loans into 5 parts, keeping **all loans of one customer in the same part** (so the test never "recognises" a customer it has already seen).
2. Learn from 4 parts: for each value of the column, what share of loans went unpaid? (Numbers are first sorted into 20 equal-sized bands, e.g. the smallest 5% of amounts, the next 5%, ...)
3. Use those shares as predictions on the 5th part, which was hidden while learning.
4. Repeat so every part is hidden once, then score all the predictions.

This is the same as asking: *"if all you knew was this one column, how well could you guess?"*

**The two scores:**
- **ROC-AUC:** 0.5 = coin flip, 1.0 = perfect. Above about **0.9 is a red flag**.
- **PR-AUC:** how well the column finds the rare defaulters. Guessing at random scores about **0.018** (the share of defaulters), and 1.0 is perfect.

We leave out the 190 not-yet-due loans (Step 3). `price` (repay ÷ amount, from Step 4) is included for comparison.

In [17]:
from sklearn.model_selection import GroupKFold
from sklearn.metrics import average_precision_score

audit = train[~not_yet_due].reset_index(drop=True)
y = audit.target
folds = list(GroupKFold(n_splits=5).split(audit, y, groups=audit.customer_id))

def as_groups(col):
    # Text columns and name-tag columns keep their values; numbers and dates are cut into 20 bands
    values = audit[col]
    if col in ['ID', 'customer_id', 'tbl_loan_id', 'lender_id'] or not pd.api.types.is_numeric_dtype(values) and not pd.api.types.is_datetime64_any_dtype(values):
        return values.astype(str)
    if pd.api.types.is_datetime64_any_dtype(values):
        values = values.map(pd.Timestamp.toordinal)
    return pd.qcut(values, 20, duplicates='drop').astype(str)

def one_column_scores(col):
    groups = as_groups(col)
    predictions = pd.Series(0.0, index=audit.index)
    for learn, hidden in folds:
        unpaid_share = y[learn].groupby(groups[learn]).mean()
        # values never seen while learning get the overall unpaid share
        predictions[hidden] = groups[hidden].map(unpaid_share).fillna(y[learn].mean()).values
    return roc_auc_score(y, predictions), average_precision_score(y, predictions)

columns = [c for c in audit.columns if c != 'target']
power = pd.DataFrame([one_column_scores(c) for c in columns], index=columns, columns=['ROC-AUC', 'PR-AUC'])
power['red flag?'] = np.where(power['ROC-AUC'] > 0.9, 'YES', '')
power.sort_values('ROC-AUC', ascending=False).round(3)

,ROC-AUC,PR-AUC,red flag?
price,0.971,0.308,YES
Lender_portion_Funded,0.745,0.065,
loan_type,0.745,0.159,
due_date,0.732,0.059,
disbursement_date,0.731,0.058,
Lender_portion_to_be_repaid,0.710,0.045,
Amount_Funded_By_Lender,0.690,0.042,
Total_Amount_to_Repay,0.669,0.040,
Total_Amount,0.660,0.037,
lender_id,0.655,0.096,


**What this means:**

1. **Only `price` is a red flag (0.97).** That confirms Step 4: the repay amount contains penalties added after default.
2. **`Total_Amount_to_Repay` *on its own* scores only 0.67, yet it is the leak.** The leak only shows once you divide it by the loan amount, because big loans owe big amounts whether or not penalties were added. **Lesson: this test cannot catch every leak by itself.** That is why we also ran the maths check and the defaulters-vs-repayers comparison.
3. **The two suspects (`Lender_portion_Funded` 0.75, `Amount_Funded_By_Lender` 0.69) are not red flags.** They predict about as well as `loan_type`, an ordinary clue. So there is no proof they peek at the answer. They stay as "contested", and Step 6 tests the model with and without them.
4. **Name tags (`ID`, `customer_id`, `tbl_loan_id`) and `country_id` score 0.49, a coin flip.** Every hidden loan belongs to a customer the test has never seen, so the name tag is useless. Country is "Kenya" on every row. This backs up dropping them as clues.
5. **The dates score 0.73.** This is not a leak: the share of unpaid loans changed over time (Step 3), so *when* a loan was sent tells you something. Because the two dates score almost the same, `due_date` adds nothing beyond `disbursement_date`.

## Step 6: Could a column secretly reveal the country?

**Why this matters:** the model is trained on Kenya only, but will be used in Ghana. For fairness we must check whether Ghanaian applicants are treated differently. If a column (say `lender_id`) is almost always different in Ghana, the model could treat "this lender" as a stand-in for "Ghanaian", even though we never give it the country. A column that quietly stands in for a group is called a **proxy**.

**How we test it:** `Test.csv` has loans from both countries. We use the same one-column test as Step 5, but the thing being guessed is now "is this loan from Ghana?" instead of "did they default?".
- ROC-AUC near **0.5**: the column looks the same in both countries (not a proxy).
- ROC-AUC near **1.0**: the column gives the country away (a strong proxy).

In [18]:
test = pd.read_csv(DATA + 'Test.csv', parse_dates=['disbursement_date', 'due_date'])
is_ghana = (test.country_id == 'Ghana').astype(int)
print('Test loans:', len(test), '| from Ghana:', is_ghana.sum(), f'({is_ghana.mean():.0%})')

test_folds = list(GroupKFold(n_splits=5).split(test, is_ghana, groups=test.customer_id))

def country_score(col):
    values = test[col]
    if col in ['ID', 'customer_id', 'tbl_loan_id', 'lender_id'] or not (pd.api.types.is_numeric_dtype(values) or pd.api.types.is_datetime64_any_dtype(values)):
        groups = values.astype(str)
    else:
        if pd.api.types.is_datetime64_any_dtype(values):
            values = values.map(pd.Timestamp.toordinal)
        groups = pd.qcut(values, 20, duplicates='drop').astype(str)
    predictions = pd.Series(0.0, index=test.index)
    for learn, hidden in test_folds:
        ghana_share = is_ghana[learn].groupby(groups[learn]).mean()
        predictions[hidden] = groups[hidden].map(ghana_share).fillna(is_ghana[learn].mean()).values
    return roc_auc_score(is_ghana, predictions)

proxy = pd.Series({c: country_score(c) for c in test.columns if c not in ['ID', 'country_id']}, name='ROC-AUC (Ghana vs Kenya)')
proxy.sort_values(ascending=False).round(3).to_frame()

Test loans: 18594 | from Ghana: 3525 (19%)


,ROC-AUC (Ghana vs Kenya)
loan_type,1.000
lender_id,1.000
Lender_portion_Funded,0.940
Lender_portion_to_be_repaid,0.786
Amount_Funded_By_Lender,0.784
due_date,0.772
disbursement_date,0.755
Total_Amount_to_Repay,0.742
Total_Amount,0.736
duration,0.550


**What this means:**
- **`loan_type` and `lender_id` give the country away perfectly (1.0).** Ghana and Kenya use different lenders and different loan types, so knowing either one tells you the country for certain. They are **strong proxies for country**.
- **`Lender_portion_Funded` is close behind (0.94):** Ghanaian lenders fund their share of loans differently.
- **The money columns and dates (0.74–0.79)** differ partly because the amounts are in different currencies (cedis vs shillings, see `docs/01_data.md`) and the countries' loans were sent at different times.
- **`duration` (0.55) and `New_versus_Repeat` (0.49) look the same in both countries.** These clues should carry over from Kenya to Ghana most safely.

### Do Ghana's lenders and loan types even appear in the Kenyan training data?

If a Ghanaian lender or loan type never appears in training, the model has learned nothing about it.

In [19]:
for col in ['lender_id', 'loan_type']:
    ghana_values = test.loc[is_ghana == 1, col].value_counts()
    seen = ghana_values.index.isin(train[col].unique())
    print(f'{col}: Ghana uses {len(ghana_values)} values; {seen.sum()} of them appear in Kenyan training data')
    print(pd.DataFrame({'Ghana loans': ghana_values, 'in training?': np.where(seen, 'yes', 'NO')}).to_string(), '\n')

lender_id: Ghana uses 5 values; 1 of them appear in Kenyan training data
           Ghana loans in training?
lender_id                          
296542            1803           NO
297183            1264           NO
296540             179           NO
297182             163           NO
245684             116          yes 

loan_type: Ghana uses 3 values; 1 of them appear in Kenyan training data
           Ghana loans in training?
loan_type                          
Type_3            3039           NO
Type_2             449          yes
Type_8              37           NO 



**What this means:** almost every Ghanaian loan has a lender and a loan type the model **has never seen**. Only 116 of 3,525 Ghanaian loans use a lender from training, and only 449 use a loan type from training. For everyone else, whatever the model learned about lenders and loan types from Kenya simply doesn't apply.

**Decision:** `lender_id` and `loan_type` are known at decision time and are not leaks, so they stay as **features**. But:
1. Step 7 tests the model **with and without** them, to see how much they really add.
2. The Ghana transfer study (Alice, step 10) must decide how to treat lenders and loan types the model has never seen.
3. The fairness work (step 11) must remember they stand in for country.

## Step 7: Build the model with different sets of columns, and compare

**What we are doing:** training a real prediction model several times, each time giving it a different set of columns. Then we compare the scores.

**Why:** for columns we are unsure about, we don't decide by opinion. We measure what they add. And one set deliberately includes the leaky columns, to show how much they would have faked the results.

### The sets of columns

| Set | Columns | Why we test it |
|---|---|---|
| **F0 Strict** | `loan_type`, `duration`, `New_versus_Repeat`, `lender_id`, calendar parts of `disbursement_date` (month, day of week, day of month) | Only columns that are clearly safe. **The default choice.** |
| **F0 − lender** | F0 without `lender_id` | `lender_id` is a country proxy (Step 6). What does it add? |
| **F0 − lender − loan type** | F0 without `lender_id` and `loan_type` | Both are country proxies that Ghana mostly doesn't share. Only clues that carry over to Ghana. |
| **F1 + amount** | F0 + loan amount | Is the amount asked for a useful clue? (Task 2 only, since the amount *is* the Task 1 answer) |
| **F2 + lender funding** | F1 + `Amount_Funded_By_Lender`, `Lender_portion_Funded` | The two "contested" columns (Step 4). |
| **F3 Everything** | every column except name tags, **including the leaks** | **Never used for real.** Shows how much leaks would fake the score. |

We don't use the year: a model in use will always meet years it has never seen.

### The two models

- **Logistic regression:** a simple, explainable model. It adds up a score from each clue. `class_weight="balanced"` makes it pay extra attention to the rare defaulters.
- **LightGBM:** a more powerful model that builds many small decision trees.

Both use the same 5 customer-grouped parts as Step 5, so a customer is never in training and testing at the same time.

**How to read the scores:** PR-AUC (random guessing ≈ 0.018) and ROC-AUC (0.5 = coin flip). Higher is better for both.

In [20]:
import warnings
from lightgbm import LGBMClassifier, LGBMRegressor
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
warnings.filterwarnings('ignore', message='X does not have valid feature names')

# Build every column we might use, from the loans in `audit` (not-yet-due loans already removed)
X_all = pd.DataFrame({
    'loan_type': audit.loan_type,
    'lender_id': audit.lender_id.astype(str),
    'New_versus_Repeat': audit.New_versus_Repeat,
    'month': audit.disbursement_date.dt.month.astype(str),
    'day_of_week': audit.disbursement_date.dt.dayofweek.astype(str),
    'day_of_month': audit.disbursement_date.dt.day,
    'log_duration': np.log1p(audit.duration),
    'log_amount': np.log1p(audit.Total_Amount),
    'log_funded': np.log1p(audit.Amount_Funded_By_Lender),
    'share_funded': audit.Lender_portion_Funded,
    'log_repay': np.log1p(audit.Total_Amount_to_Repay),
    'log_to_be_repaid': np.log1p(audit.Lender_portion_to_be_repaid),
    'price': audit.Total_Amount_to_Repay / audit.Total_Amount,
})
TEXT = ['loan_type', 'lender_id', 'New_versus_Repeat', 'month', 'day_of_week']

F0 = ['loan_type', 'lender_id', 'New_versus_Repeat', 'month', 'day_of_week', 'day_of_month', 'log_duration']
feature_sets = {
    'F0 Strict': F0,
    'F0 − lender': [c for c in F0 if c != 'lender_id'],
    'F0 − lender − loan type': [c for c in F0 if c not in ('lender_id', 'loan_type')],
    'F1 + amount': F0 + ['log_amount'],
    'F2 + lender funding': F0 + ['log_amount', 'log_funded', 'share_funded'],
    'F3 Everything (leaks)': list(X_all.columns),
}

def prepare(cols, for_linear):
    # Text columns become yes/no columns (linear) or numbered categories (trees); numbers are scaled for the linear model
    text = [c for c in cols if c in TEXT]
    numbers = [c for c in cols if c not in TEXT]
    if for_linear:
        return ColumnTransformer([('text', OneHotEncoder(handle_unknown='ignore'), text),
                                  ('numbers', StandardScaler(), numbers)])
    return ColumnTransformer([('text', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), text),
                              ('numbers', 'passthrough', numbers)])

def classify_scores(cols, model_name):
    predictions = np.zeros(len(audit))
    for learn, hidden in folds:
        if model_name == 'Logistic regression':
            model = make_pipeline(prepare(cols, True), LogisticRegression(class_weight='balanced', max_iter=2000))
        else:
            model = make_pipeline(prepare(cols, False), LGBMClassifier(n_estimators=300, learning_rate=0.05, verbose=-1, random_state=0))
        model.fit(X_all.loc[learn, cols], y[learn])
        predictions[hidden] = model.predict_proba(X_all.loc[hidden, cols])[:, 1]
    return average_precision_score(y, predictions), roc_auc_score(y, predictions)

### Task 2: predicting default

In [21]:
rows = {}
for set_name, cols in feature_sets.items():
    for model_name in ['Logistic regression', 'LightGBM']:
        rows[(set_name, model_name)] = classify_scores(cols, model_name)
task2 = pd.DataFrame(rows, index=['PR-AUC', 'ROC-AUC']).T
task2.loc[('Baseline: guess the same for everyone', ''), :] = [y.mean(), 0.5]
task2.round(3)

PR-AUC  ROC-AUC
F0 Strict                             Logistic regression   0.200    0.794
                                      LightGBM              0.170    0.773
F0 − lender                           Logistic regression   0.199    0.795
                                      LightGBM              0.169    0.775
F0 − lender − loan type               Logistic regression   0.151    0.728
                                      LightGBM              0.148    0.756
F1 + amount                           Logistic regression   0.200    0.797
                                      LightGBM              0.168    0.796
F2 + lender funding                   Logistic regression   0.201    0.818
                                      LightGBM              0.166    0.815
F3 Everything (leaks)                 Logistic regression   0.749    0.990
                                      LightGBM              0.913    0.995
Baseline: guess the same for everyone                       0.018    0.500

**What this means:**
1. **Leaks fake the results massively.** With the leaky columns (F3), PR-AUC jumps from about **0.20 to 0.91**, more than four times higher. A team that missed the leak would report a nearly perfect model that would fail completely in real use. The difference (about 0.7) is **how much leakage would have inflated our results**.
2. **`lender_id` adds nothing** (0.200 with it vs 0.199 without). Since it is also a country proxy that Ghana mostly doesn't share, dropping it costs nothing.
3. **`loan_type` does matter** (PR-AUC falls from 0.20 to 0.15 without it). But Ghana mostly uses loan types the model has never seen, so this help may not carry over to Ghana.
4. **The amount and the two contested funding columns add almost nothing** for default (PR-AUC 0.200 → 0.201).
5. **The simple model (logistic regression) does as well as LightGBM, or better.** This matches what the brief expects: careful, leak-free columns matter more than a fancy model.
6. **Every honest set beats the baseline by far** (0.15–0.20 vs 0.018), so these columns do carry real information.

### Task 1: predicting the loan amount

The answer is the loan amount, so no set may contain the amount or anything calculated from it. We predict the **log** of the amount, because amounts range from 2 to 23 million and the log stops a few huge loans from dominating.

**Score: MAE of log amount**, the average size of the error. Lower is better. An MAE of 0.5 means the guess is typically off by a factor of about 1.6 (e^0.5).

The last set adds the lender-funding columns, which are calculated from the amount (Step 4), to show what that leak would do.

In [22]:
log_amount = X_all.log_amount
task1_sets = {
    'F0 Strict': F0,
    'F0 − lender': feature_sets['F0 − lender'],
    'F0 − lender − loan type': feature_sets['F0 − lender − loan type'],
    'F0 + lender funding (leak for Task 1)': F0 + ['log_funded', 'share_funded'],
}

def regress_mae(cols, model_name):
    predictions = np.zeros(len(audit))
    for learn, hidden in folds:
        if model_name == 'Linear regression (Ridge)':
            model = make_pipeline(prepare(cols, True), Ridge())
        else:
            model = make_pipeline(prepare(cols, False), LGBMRegressor(n_estimators=300, learning_rate=0.05, verbose=-1, random_state=0))
        model.fit(X_all.loc[learn, cols], log_amount[learn])
        predictions[hidden] = model.predict(X_all.loc[hidden, cols])
    return np.abs(predictions - log_amount).mean()

rows = {(s, m): regress_mae(cols, m) for s, cols in task1_sets.items()
        for m in ['Linear regression (Ridge)', 'LightGBM']}
# Baseline: in each hidden part, guess the average log amount of the learning parts
baseline_guess = np.zeros(len(audit))
for learn, hidden in folds:
    baseline_guess[hidden] = log_amount[learn].mean()
baseline = np.abs(baseline_guess - log_amount).mean()
task1 = pd.Series(rows, name='MAE of log amount').to_frame()
task1.loc[('Baseline: guess the average for everyone', ''), :] = baseline
task1.round(3)

MAE of log amount
F0 Strict                                Linear regression (Ridge)              0.911
                                         LightGBM                               0.910
F0 − lender                              Linear regression (Ridge)              0.911
                                         LightGBM                               0.910
F0 − lender − loan type                  Linear regression (Ridge)              0.923
                                         LightGBM                               0.913
F0 + lender funding (leak for Task 1)    Linear regression (Ridge)              0.631
                                         LightGBM                               0.141
Baseline: guess the average for everyone                                        0.977

**What this means:**
1. **Predicting the loan amount from the safe columns is hard.** The best honest set (MAE 0.91) is only a little better than guessing the average for everyone (0.98). A typical guess is off by a factor of about 2.5. The safe columns say little about *how much* someone borrows. Borrower history (step 5, e.g. "how much did they borrow last time?") should help a lot.
2. **The funding columns leak the amount.** Adding them drops the error from 0.91 to **0.14** with LightGBM. That isn't skill: `funded` = share × amount, so the model just works the amount back out.
3. **`lender_id` adds nothing here either,** and `loan_type` adds only a little.

**Decision from Step 7:**
- **Deployment set = F0 − lender:** `loan_type`, `duration`, `New_versus_Repeat`, and calendar parts of `disbursement_date`. It scores the same as F0 without using a country proxy.
- `Total_Amount` is the **Task 1 answer**. It is **not** used for Task 2 either, because it adds nothing there and the brief asks for the same columns in both tasks.
- `Amount_Funded_By_Lender` and `Lender_portion_Funded` are **dropped**: they leak the Task 1 answer and add nothing to Task 2.

## Step 8: The final feature audit table

One row per column of `Train.csv`. This is the table the brief asks for (Part A.1c); the write-up is in `learnings/03_feature_audit.md`.

"Dropped" means *not given to the model as a clue*. Dropped columns can still be kept in the data for other jobs (labelling rows, grouping customers, fairness checks).

In [23]:
final_audit = pd.DataFrame([
    ('ID', 'yes', 'dropped', 'Customer + loan + lender glued together (100% of rows); unique per row so no pattern to learn. Kept only to match predictions for Zindi.'),
    ('customer_id', 'yes', 'dropped', 'Name tag (one-column test: 0.49, a coin flip). Used to keep each customer in one test fold and to build borrower history.'),
    ('country_id', 'yes', 'dropped', 'Kenya on every training row, so nothing to learn. Kept for fairness checks.'),
    ('tbl_loan_id', 'yes', 'dropped', 'Name tag. Used to group the rows of one loan shared between lenders.'),
    ('lender_id', 'yes', 'dropped', 'Adds nothing (PR-AUC 0.200 with vs 0.199 without) and reveals the country (1.00). Kept for fairness checks.'),
    ('loan_type', 'yes', 'feature', 'Chosen at application; strongest safe clue (PR-AUC 0.20 with, 0.15 without). Reveals country (1.00): check in fairness.'),
    ('Total_Amount', 'yes', 'target (Task 1)', 'The loan amount to predict. Not used for Task 2: adds nothing (0.200 → 0.200) and both tasks share one feature set.'),
    ('Total_Amount_to_Repay', 'NO', 'dropped (leak)', 'Penalties added after default: same product, repayers owe ~1% extra, defaulters ~41%; repay ÷ amount ROC-AUC 0.99.'),
    ('disbursement_date', 'yes', 'feature (calendar parts)', 'The decision moment. Used as month, day of week, day of month; not the year (future years are always unseen).'),
    ('due_date', 'yes', 'dropped', 'Always disbursement_date + duration (100% of rows), so it adds nothing new.'),
    ('duration', 'yes', 'feature', 'Agreed before the money is sent; longer loans default far more (1.2% at ≤7 days, 44.7% at >90). Same in both countries (0.55).'),
    ('New_versus_Repeat', 'yes', 'feature', "Known from the customer's record; same in both countries (0.49). Also a fairness group (only 567 'New' loans)."),
    ('Amount_Funded_By_Lender', 'yes', 'dropped', 'Calculated from the amount (share × amount), so it leaks the Task 1 answer (MAE 0.91 → 0.14); adds nothing to Task 2.'),
    ('Lender_portion_Funded', 'yes', 'dropped', 'Exactly funded ÷ amount (100% of rows): leaks the Task 1 answer, adds nothing to Task 2.'),
    ('Lender_portion_to_be_repaid', 'NO', 'dropped (leak)', "This lender's slice of Total_Amount_to_Repay (96% of rows), so it carries the same penalty leak."),
    ('target', 'NO', 'target (Task 2)', '1 = not repaid as agreed by data collection (~mid-Nov 2024). 190 loans not yet due are left out of training.'),
], columns=['column', 'known at decision time?', 'role', 'reason (evidence)']).set_index('column')

assert set(final_audit.index) == set(train.columns.drop('price')), 'every Train.csv column needs exactly one row'
pd.set_option('display.max_colwidth', 200)
final_audit

,known at decision time?,role,reason (evidence)
column,,,
ID,yes,dropped,Customer + loan + lender glued together (100% of rows); unique per row so no pattern to learn. Kept only to match predictions for Zindi.
customer_id,yes,dropped,"Name tag (one-column test: 0.49, a coin flip). Used to keep each customer in one test fold and to build borrower history."
country_id,yes,dropped,"Kenya on every training row, so nothing to learn. Kept for fairness checks."
tbl_loan_id,yes,dropped,Name tag. Used to group the rows of one loan shared between lenders.
lender_id,yes,dropped,Adds nothing (PR-AUC 0.200 with vs 0.199 without) and reveals the country (1.00). Kept for fairness checks.
loan_type,yes,feature,"Chosen at application; strongest safe clue (PR-AUC 0.20 with, 0.15 without). Reveals country (1.00): check in fairness."
Total_Amount,yes,target (Task 1),The loan amount to predict. Not used for Task 2: adds nothing (0.200 → 0.200) and both tasks share one feature set.
Total_Amount_to_Repay,NO,dropped (leak),"Penalties added after default: same product, repayers owe ~1% extra, defaulters ~41%; repay ÷ amount ROC-AUC 0.99."
disbursement_date,yes,feature (calendar parts),"The decision moment. Used as month, day of week, day of month; not the year (future years are always unseen)."


### The frozen column list for step 5 (shared features)

These are the only raw columns the models may use. Borrower history and macro features are added on top in step 5, and this comparison is rerun then.

In [24]:
FEATURES = ['loan_type', 'duration', 'New_versus_Repeat', 'disbursement_date']  # disbursement_date → month, day of week, day of month
TARGET_TASK1 = 'Total_Amount'
TARGET_TASK2 = 'target'
FAIRNESS_GROUPS = ['country_id', 'loan_type', 'lender_id', 'New_versus_Repeat']  # plus loan-size band
LEAVE_OUT_OF_TRAINING = 'loans whose due_date is after the last disbursement_date (labels not known yet)'
print('Features:', FEATURES)

Features: ['loan_type', 'duration', 'New_versus_Repeat', 'disbursement_date']
